# Train an Eldertide agent

PPO with action masking and self-play, on the simulator in `rl/`.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
2. Training saves to Google Drive every 10 minutes. If the session ends, reconnect and run the
   cells again: the training cell resumes where it stopped.
3. Keep this tab open while it trains (free Colab disconnects idle sessions).

In [ ]:
# What this runtime has (free Colab: about 2 CPU cores and a T4).
import os, subprocess
print("CPU cores:", os.cpu_count())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU: pick a T4 runtime")

In [ ]:
# Google Drive keeps checkpoints and metrics between sessions.
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = "/content/drive/MyDrive/eldertide/run1"  # change the name to start a separate run
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
# Get (or update) the code.
REPO = "https://github.com/SanketJadhav7d3/eldertide.git"
if not os.path.exists("/content/eldertide"):
    !git clone --depth 1 {REPO} /content/eldertide
else:
    !git -C /content/eldertide pull --ff-only
%cd /content/eldertide/rl
!pip install -q -r requirements.txt

## Check

The simulator's tests (about a minute), then a tiny training run that exercises the network, the
game workers, the PPO update, checkpointing and resuming. Both should pass before the real run.

In [ ]:
!python -m pytest -q tests -x

In [ ]:
!rm -rf /content/smoke
!python -m eldertide_rl.train.ppo --smoke --run-dir /content/smoke
!python -m eldertide_rl.train.ppo --smoke --run-dir /content/smoke --total-env-steps 192  # resumes and finishes
!python -m eldertide_rl.train.evaluate --run-dir /content/smoke --games 2

## Train

Runs until `--total-env-steps` (30 million decisions by default) or until you stop the cell; stopping
saves a checkpoint. Run it again to continue. Other settings: `python -m eldertide_rl.train.ppo --help`.

In [ ]:
!python -m eldertide_rl.train.ppo --run-dir {RUN_DIR}

## Progress

`win_rush`, `win_random` and `win_past` are the learner's score (win 1, draw ½, loss 0) over its last
100 games against each opponent.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

m = pd.read_csv(f"{RUN_DIR}/metrics.csv")
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
m.plot(x="env_steps", y=["win_random", "win_rush", "win_past"], ax=ax[0, 0], title="Score vs opponents", ylim=(0, 1))
m.plot(x="env_steps", y="mean_game_decisions", ax=ax[0, 1], title="Game length (decisions)")
m.plot(x="env_steps", y="entropy", ax=ax[1, 0], title="Policy entropy")
m.plot(x="env_steps", y=["approx_kl", "clip_fraction"], ax=ax[1, 1], title="Update size")
plt.tight_layout()
m.tail()

## Evaluate

Plays the latest checkpoint against RandomBot and RushBot.

In [ ]:
!python -m eldertide_rl.train.evaluate --run-dir {RUN_DIR} --games 10